In [1]:
import pandas as pd

In [2]:
faq_df = pd.read_csv("../data/faq_seed.csv")

In [3]:
faq_df.head()

,id,question,answer
0,0,How do I get a refund?,"To request a refund, visit your orders page an..."
1,1,Can I reset my password?,Click **Forgot Password** on the login page an...
2,2,Where is my order?,Use the tracking link sent to your email after...
3,3,How long is the warranty?,All electronic products include a 12-month war...
4,4,Do you ship mternationally?,"Yes, we ship to over 50 countries worldwide. I..."


In [4]:
# import sentence_transformers from huggingface for encoding the questions to get their vector representations
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
# Encode the questions to get their vector representations
faq_embeddings = encoder.encode(faq_df["question"].tolist())

print(f"Sample (First 10 dimensions): {faq_embeddings[0][:10]}")

Sample (First 10 dimensions): [-0.05662674  0.00819996  0.04742412  0.03048575  0.00012446 -0.01017602
  0.01872805  0.00375811 -0.01293179  0.03286451]


In [6]:
import numpy as np

In [7]:
def cosine_dist(a: np.array, b: np.array):
    """Compute cosine distance between two sets of vectors."""
    a_norm = np.linalg.norm(a, axis=1)
    b_norm = np.linalg.norm(b) if b.ndim == 1 else np. linalg.norm(b, axis=1)
    sim = np.dot(a, b) / (a_norm * b_norm)
    return 1 - sim

In [8]:
def semantic_search(query: str)->tuple:
    """Find the most similar FAQ question to the query."""
    query_embedding = encoder.encode([query]) [0]

    distances = cosine_dist(faq_embeddings, query_embedding)
    
    #find the most similar question(lowest distance)
    best_idx = np.argmin(distances)
    best_distance = distances[best_idx]

    return best_idx, best_distance

In [9]:
idx, distance = semantic_search("How long will it take to get a refund for my order?")

In [10]:
idx, distance = semantic_search("when will i get my refund?")

In [11]:
print(f"Most similar FAQ: {faq_df.iloc[idx]['question']}")
print(f"Answer: {faq_df.iloc[idx]['answer']}")
print(f"Cosine Distance: {distance:.3f}")

Most similar FAQ: How do I get a refund?
Answer: To request a refund, visit your orders page and select **Request Refund**. Refunds are processed within 3-5 business days.
Cosine Distance: 0.242


In [12]:
def check_cache(query: str, distance_threshold: float = 0.3):
    """
    Semantic cache lookup for previously asked questions.
    Returns a dictionary with answer if hit, None if miss.
    """
    idx, distance = semantic_search(query)

    if distance <= distance_threshold:
        return {
            "prompt": faq_df.iloc[idx]["question"],
            "response": faq_df.iloc[idx]["answer"],
            "vector_distance": float(distance),
        }

    return None  # Cache miss

In [13]:
test_queries = [
    "Is it possible to get a refund?",
    "I want my money back",
    "What are your business hours?",  # Should miss
]

for query in test_queries:
    result = check_cache(query, distance_threshold=0.25)
    if result:
        print(f"✅ HIT: '{query}' -> {result['response'][:50]}...")
        print(f"   Distance: {result['vector_distance']:.3f}\n")
    else:
        print(f"❌ MISS: '{query}'\n")

✅ HIT: 'Is it possible to get a refund?' -> To request a refund, visit your orders page and se...
   Distance: 0.160

❌ MISS: 'I want my money back'

❌ MISS: 'What are your business hours?'



In [14]:
def add_to_cache(question: str, answer: str):
    """
    Add a new Q&A pair to our simple in-memory cache.
    Extends both the DataFrame and embeddings matrix.
    """
    global faq_df, faq_embeddings

    new_row = pd.DataFrame({"question": [question], "answer": [answer]})
    faq_df = pd.concat([faq_df, new_row], ignore_index=True)

    # Generate embedding for the new question
    new_embedding = encoder.encode([question])

    # Add to embeddings matrix
    faq_embeddings = np.vstack([faq_embeddings, new_embedding])

    print(f"✅ Added to cache: '{question}'")

In [15]:
print("Original cache size:", len(faq_df))

new_entries = [
    (
        "What are your business hours?",
        "We're open Monday-Friday 9 AM to 6 PM EST. Weekend support is available for urgent issues.",
    ),
    (
        "Do you have a mobile app?",
        "Yes! Our mobile app is available on both iOS and Android. Search for 'CustomerApp' in your app store.",
    ),
    (
        "How do I update my payment method?",
        "Go to Account Settings > Payment Methods to add, edit, or remove payment options.",
    ),
]

for question, answer in new_entries:
    add_to_cache(question, answer)

print(f"\nCache now has {len(faq_df)} total entries")

Original cache size: 8
✅ Added to cache: 'What are your business hours?'
✅ Added to cache: 'Do you have a mobile app?'
✅ Added to cache: 'How do I update my payment method?'

Cache now has 11 total entries


In [16]:
test_extended_queries = [
    "What time do you open?",  
    "Is there a phone app?", 
    "How can I change my payment method?",
]

for query in test_extended_queries:
    result = check_cache(query, distance_threshold=0.3)
    if result:
        print(f"✅ HIT: '{query}' -> {result['response'][:50]}...")
        print(f"   Distance: {result['vector_distance']:.3f}\n")
    else:
        print(f"❌ MISS: '{query}'\n")

❌ MISS: 'What time do you open?'

✅ HIT: 'Is there a phone app?' -> Yes! Our mobile app is available on both iOS and A...
   Distance: 0.215

✅ HIT: 'How can I change my payment method?' -> Go to Account Settings > Payment Methods to add, e...
   Distance: 0.107



In [17]:
import os 

REDIS_URL = os.getenv("REDIS_URL", "redis://localhost:6379")

In [18]:
import redis

In [19]:
try:
    r: redis.Redis = redis.from_url(REDIS_URL)
    r.ping()
    print("✅ Redis is running and accessible.")
except redis.ConnectionError:
    print("❌ Redis is not accessible. Please check your Redis server.")
    raise


✅ Redis is running and accessible.


In [20]:
from redisvl.utils.vectorize import HFTextVectorizer
from redisvl.extensions.cache.embeddings import EmbeddingsCache

langcache_embed = HFTextVectorizer(
    model="sentence-transformers/all-MiniLM-L6-v2",
    cache=EmbeddingsCache(redis_client=r, ttl=3600)#ttl means time to live for the cache, in seconds. Here it's set to 1 hour (3600 seconds)    
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [21]:
from redisvl.extensions.cache.llm import SemanticCache

cache = SemanticCache(
    name="faq-cache-v1",
    vectorizer=langcache_embed,
    redis_client=r,
    distance_threshold=0.3
)

In [22]:
for i in range(len(faq_df)):
    cache.store(
        prompt=faq_df.iloc[i]["question"],
        response=faq_df.iloc[i]["answer"],
    )

In [23]:
cache

SemanticCache(name='faq-cache-v1', distance_threshold=0.3, ttl=None)

In [24]:
result = cache.check("when will i get my refund?")


In [25]:
result

[{'entry_id': '60fd55b8527fcd2bf427d81dc3f4c47c4bf8904c9802ffecbcf2c02b38f537ac',
  'prompt': 'How do I get a refund?',
  'response': 'To request a refund, visit your orders page and select **Request Refund**. Refunds are processed within 3-5 business days.',
  'vector_distance': 0.242215931416,
  'inserted_at': 1782747210.88,
  'updated_at': 1782747210.88,
  'key': 'faq-cache-v1:60fd55b8527fcd2bf427d81dc3f4c47c4bf8904c9802ffecbcf2c02b38f537ac'}]

In [26]:
cache.set_ttl(86400)

In [28]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
import os

#set ur groq api key
os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY_HERE"

MODEL_NAME = "openai/gpt-oss-20b"

llm = ChatGroq(
    model_name=MODEL_NAME,
    temperature=0.1,
    
)

In [29]:
def get_llm_response(question: str) -> str:
    prompt = f"""
    You are a helpful customer support assistant. Answer this customer question concisely and professionally:
    
    Question: {question}
    
    Provide a helpful response in 1-2 sentences. If you don't have specific information, give a general helpful response.
    """
    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content.strip()

In [30]:
get_llm_response("When will I get my refund?")

'Refunds usually process within 5–10 business days after the request is submitted. You can check the status in your account or contact support for a specific timeline.'

In [31]:
from evals import PerfEval

perf_eval = PerfEval()

In [32]:
test_questions = [
    "How can I get my money back?",
    "I want a refund please",
    "What's your return policy?",
    "I forgot my password",
    "Can you help me reset my password?",
    "What are your shipping costs?",
    "Do you offer installation services?",
    "Can I schedule a phone call with support?",
    "How do I cancel my subscription?",
    "How much does shipping cost?",
    "I need to cancel my account",
]

In [33]:
with perf_eval:
    for i, question in enumerate(test_questions, 1):
        print(f"\n[{i}] Question: '{question}'")

        perf_eval.start()

        if cached_result := cache.check(question):
            # Cache HIT
            perf_eval.tick("cache_hit")
            print(
                f"    ✅ CACHE HIT (distance: {cached_result[0]['vector_distance']:.3f})"
            )
            print(f"    📋 Cached question: {cached_result[0]['prompt'][:80]}...")
            print(f"    📋 Cached response: {cached_result[0]['response'][:80]}...")
        else:
            # Cache MISS - call LLM
            perf_eval.tick("cache_miss")  # Time for cache check
            print(f"    ❌ CACHE MISS")
            print(f"    🤖 Calling LLM... ", end="")

            # Call LLM and track the call
            perf_eval.start()
            llm_response = get_llm_response(question)
            perf_eval.tick("llm_call")
            print(f"    💬 LLM response: {llm_response[:80]}...")
            cache.store(prompt=question, response=llm_response)


[1] Question: 'How can I get my money back?'
    ✅ CACHE HIT (distance: 0.272)
    📋 Cached question: How do I get a refund?...
    📋 Cached response: To request a refund, visit your orders page and select **Request Refund**. Refun...

[2] Question: 'I want a refund please'
    ✅ CACHE HIT (distance: 0.197)
    📋 Cached question: How do I get a refund?...
    📋 Cached response: To request a refund, visit your orders page and select **Request Refund**. Refun...

[3] Question: 'What's your return policy?'
    ❌ CACHE MISS
    🤖 Calling LLM...     💬 LLM response: We accept returns within 30 days of purchase, provided the item is in its origin...

[4] Question: 'I forgot my password'
    ✅ CACHE HIT (distance: 0.271)
    📋 Cached question: Can I reset my password?...
    📋 Cached response: Click **Forgot Password** on the login page and follow the email instructions. C...

[5] Question: 'Can you help me reset my password?'
    ✅ CACHE HIT (distance: 0.122)
    📋 Cached question: Can I res

In [34]:
np.mean(perf_eval.durations_by_label['cache_hit'])

np.float64(0.06829820360456194)

In [35]:
np.mean(perf_eval.durations_by_label['llm_call'])

np.float64(0.7137261629104614)